# Article extraction — version re-check

Checks one pipeline version's output against the reading-order error log annotated on another
(`evaluation/reading_order_error_log.csv`, annotated on **v1**: 74 pages, 1,068 boxes, 97 errors).
Every logged box is looked up in the new version's output by its `region_id` (YOLO's boxes don't change
between versions). The rules live in `compare_versions.py`; this notebook runs them, saves the CSV and
summary, and shows the tables.

**Outcomes** (all decided by the script, no hand review)

- `fixed` — checked by the script: a *merge* box now starts an article; a *split* box no longer does;
  a *misplacement* strip box is gone; a *missed* line's quoted words are back on the page.
- `changed` — the box differs in the new version (role, front/back text cut from it, article start,
  title or series), but no script can tell whether it now matches the log (mostly *role* errors).
- `unchanged` — the box is as in the old version: the error persists.
- Rows logged ok: `unchanged`, `renamed` (only the role's name changed, e.g. byline → front_matter),
  or `changed` (a possible regression).

**Caveat:** the v2 rules were written from these same pages, so this shows the decisions were carried
out — not how v2 does on unseen text (that comes from the test issues, roadmap step 5).

In [1]:
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.max_rows", 200)

# Notebook lives in evaluation/, but work from either the repo root or evaluation/.
EVAL_DIR = next(p for p in [Path("."), Path("evaluation")] if (p / "compare_versions.py").exists())
sys.path.insert(0, str(EVAL_DIR.resolve()))
from compare_versions import MAGAZINES, compare, summarise

OLD, NEW = "v1", "v2"
LOG_PATH = EVAL_DIR / "reading_order_error_log.csv"
OUT_PATH = EVAL_DIR / f"reading_order_{OLD}_vs_{NEW}.csv"

## Run the comparison and save the CSV and summary

In [2]:
log = pd.read_csv(LOG_PATH, dtype=str, keep_default_na=False)
df = compare(OLD, NEW, log)
df.to_csv(OUT_PATH, index=False)
summary_path = OUT_PATH.with_name(OUT_PATH.stem + "_summary.md")
summary_path.write_text(summarise(df, OLD, NEW))

errors = df[df.verdict == "error"].copy()
errors["error_type"] = errors.error_type.str.split(";").str[0].str.strip()
ok = df[df.verdict == "ok"]
OUTCOMES = ["fixed", "changed", "unchanged"]
print(f"Wrote {OUT_PATH} and {summary_path}")
print(f"{len(log)} log rows: {len(ok)} ok, {len(errors)} errors, {(df.verdict == 'excluded').sum()} excluded (duplicate pages)")

Wrote reading_order_v1_vs_v2.csv and reading_order_v1_vs_v2_summary.md
1090 log rows: 971 ok, 97 errors, 22 excluded (duplicate pages)


## 1. Logged errors — outcome in the new version

In [3]:
def by(field):
    t = pd.crosstab(errors[field], errors.outcome, margins=True, margins_name="total")
    return t.reindex(columns=[c for c in OUTCOMES if c in t.columns] + ["total"], fill_value=0)

by("error_type")

outcome,fixed,changed,unchanged,total
error_type,,,,
merge,18,0,4,22
misattached,0,2,0,2
misplacement,8,0,0,8
missed,1,0,0,1
role,0,55,6,61
split,2,1,0,3
total,29,58,10,97


In [4]:
by("magazine").reindex(MAGAZINES + ["total"])

outcome,fixed,changed,unchanged,total
magazine,,,,
penny,15,14,4,33
saturday,1,22,2,25
chambers,13,22,4,39
total,29,58,10,97


## 2. Before / after on the logged boxes

`changed` errors aren't confirmed either way, so the new version's share of correct boxes is a range:
*at least* (counting every `changed` as still wrong) to *at most* (counting every `changed` as fixed).

In [5]:
def before_after(frame):
    boxes = len(frame[frame.verdict.isin(["ok", "error"])])
    errs = frame[frame.verdict == "error"]
    regressions = (frame[frame.verdict == "ok"].outcome == "changed").sum()
    n = {o: (errs.outcome == o).sum() for o in OUTCOMES}
    return pd.Series({
        "boxes": boxes,
        f"{OLD} errors": len(errs),
        f"{OLD} % correct": round(100 * (boxes - len(errs)) / boxes, 1),
        f"{NEW} fixed (script)": n["fixed"],
        f"{NEW} changed (unconfirmed)": n["changed"],
        f"{NEW} unchanged": n["unchanged"],
        f"{NEW} ok rows changed": regressions,
        f"{NEW} % correct, at least": round(100 * (boxes - n["changed"] - n["unchanged"] - regressions) / boxes, 1),
        f"{NEW} % correct, at most": round(100 * (boxes - n["unchanged"]) / boxes, 1),
    }, dtype=object)

pd.DataFrame({m: before_after(df[df.magazine == m]) for m in MAGAZINES} | {"all": before_after(df)})

,penny,saturday,chambers,all
boxes,313,339,416,1068
v1 errors,33,25,39,97
v1 % correct,89.5,92.6,90.6,90.9
v2 fixed (script),15,1,13,29
v2 changed (unconfirmed),14,22,22,58
v2 unchanged,4,2,4,10
v2 ok rows changed,0,1,0,1
"v2 % correct, at least",94.2,92.6,93.8,93.5
"v2 % correct, at most",98.7,99.4,99.0,99.1


## 3. Rows logged ok — anything broken?

In [6]:
ok.outcome.value_counts().to_frame("rows")

,rows
outcome,
unchanged,957
renamed,13
changed,1


In [7]:
# Logged-ok rows that changed: possible regressions (an intended change is still listed here)
ok[ok.outcome == "changed"][["magazine", "page", "reading_order", "old_role", "new_role",
                             "old_start", "new_start", "new_title"]]

,magazine,page,reading_order,old_role,new_role,old_start,new_start,new_title
472,saturday,12,2,title,title,True,False,WILD SPORTS OF THE EAST


In [8]:
# Logged-ok rows whose role only changed name
ok[ok.outcome == "renamed"].groupby(["old_role", "new_role"]).size().to_frame("rows")

,,rows
old_role,new_role,
attribution,back_matter,1
byline,front_matter,3
end_note,back_matter,2
text,table,7


## 4. Errors that persist (`unchanged`)

In [9]:
errors[errors.outcome == "unchanged"][["magazine", "page", "reading_order", "error_type", "notes"]]

,magazine,page,reading_order,error_type,notes
103,penny,8,3,merge,merge: separate one-paragraph item 'Instruction and Amusement' (no head punctuation: heading words run str...
105,penny,8,5,merge,merge: separate item 'Devotion of a great Mind to its Duties.' (run-in head with lowercase words)
111,penny,8,11,merge,"merge: separate one-paragraph item (Michael Angelo anecdote), no head or rule at all; first box of the rig..."
122,penny,9,1,role,"role: caption '[Restored View of Pompeii.]' is included in YOLO's figure box (with the title 'POMPEII.'), ..."
370,saturday,4,1,role,role: article title 'HISTORY OF THE INDIAN SUPERSTITION OF JAGGANATHA (or Juggernaut.)' is inside YOLO's f...
467,saturday,11,8,role,"role: in-line title 'WHAT is true knowledge?' (heading run into the poem's text) not recognised, so the ar..."
722,chambers,4,6,role,"role: front matter (verse epigraph from an old ballad, 'The Yerl o' Wigton had three dauchters...') read a..."
1017,chambers,20,26,role,role: long front matter (editor's introduction to the letter: 'The editor has been favoured with the perus...
1018,chambers,20,27,role,"role: front matter (the letter's dateline 'Rochester, Monroe County, N. Y.') read as article text; could b..."
1081,chambers,24,4,merge,merge: separate article with in-line title 'VEGETABLES' merged into Cheese Hoppers (tilt scrambled the box...


## 5. Errors that changed (unconfirmed) — what the new version now has

In [10]:
changed = errors[errors.outcome == "changed"]
changed.assign(front_back=(changed.front_text + " / " + changed.back_text).str.strip(" /"))[
    ["magazine", "page", "reading_order", "error_type", "old_role", "new_role",
     "new_title", "new_series", "front_back", "notes"]]

,magazine,page,reading_order,error_type,old_role,new_role,new_title,new_series,front_back,notes
63,penny,6,3,role,title,front_matter,ISAAC ASHFORD,,,role: front matter (source line '[From Crabbe's Parish Register.]') boxed as a title and joined to it
137,penny,10,5,role,text,front_matter,VAN DIEMEN'S LAND,,,role: '[Concluded from our last.]' read as article text; should be part of the title / a subtitle (title_n...
156,penny,12,1,role,title,series_heading,THE DORMOUSE,British Animals,,"role: 'BRITISH ANIMALS.' is a series heading (recurring feature: Dormouse, Swallow, later the Mole), not p..."
187,penny,15,2,role,text,text,POESIE,,"[GEORGE WITHER, BORN 1588, DIED 1677.]","role: front matter '[GEORGE WITHER, BORN 1588, DIED 1677.]' read as article text (box also holds the poem)"
206,penny,16,11,role,figure,imprint,,,,"role: end-of-issue publisher's imprint ('LONDON:—CHARLES KNIGHT, PALL-MALL EAST... Printed by WILLIAM CLOW..."
221,penny,17,,role,furniture,back_matter,EMIGRATION TO THE NORTH AMERICAN COLONIES,,,"role: asterisked footnote giving the source of 'Emigration to the North American Colonies' (box 9-10), val..."
243,penny,19,9,role,text,front_matter,THUS I THINK,,,"role: front matter (source line '[From Locke's Miscellaneous Papers, published in his Life by Lord King.]'..."
258,penny,20,4,role,figure,figure,THE MOLE,British Animals,,role: series heading 'BRITISH ANIMALS.' is inside YOLO's figure box; the Mole should get series = 'British...
271,penny,21,4,role,title,front_matter,THE LIBRARY,,,"role: first line of the Shakespeare epigraph ('""My Library / Was Dukedom large enough."" —Prospero, in the ..."
303,penny,24,4,role,text,text,THE WOODMAN'S MEMORIAL,,[From the 'Plain Englishman; published in 1821-2-3.],role: front matter (source line '[From the 'Plain Englishman; published in 1821-2-3.]') read as article te...


## 6. Look up a page

In [11]:
def page(magazine, page_no):
    """Every logged box on one page, old vs new."""
    rows = df[(df.magazine == magazine) & (df.page.astype(int) == page_no)]
    return rows[["reading_order", "verdict", "error_type", "old_role", "new_role",
                 "old_start", "new_start", "new_title", "outcome"]]

page("penny", 8)

,reading_order,verdict,error_type,old_role,new_role,old_start,new_start,new_title,outcome
101,1,ok,,text,text,True,True,Power of Steam,unchanged
102,2,ok,,text,text,True,True,A Popular Error,unchanged
103,3,error,merge,text,text,False,False,A Popular Error,unchanged
104,4,error,merge,text,text,False,True,The Secret of great Workers,fixed
105,5,error,merge,text,text,False,False,The Secret of great Workers,unchanged
106,6,error,merge,text,text,False,True,,fixed
107,7,error,merge,text,text,False,True,,fixed
108,8,error,merge,text,text,False,True,,fixed
109,9,error,merge,text,text,False,True,,fixed
110,10,error,merge,text,text,False,True,,fixed
